# Verifikasi Perhitungan AIC pada Grid Search SARIMAX (Data Produk Asli)

Notebook ini melakukan verifikasi independen bahwa nilai AIC yang digunakan
untuk memilih parameter ARIMA terbaik pada `prediction.py` (Kode 4.13)
benar-benar mengikuti rumus **AIC = 2k - 2 ln(L)**.

Verifikasi dilakukan pada **satu produk asli** dari dataset Toko Loa Kim
Jong (bukan data sintetis), sehingga hasil kombinasi parameter terbaik
yang keluar di sini seharusnya **konsisten** dengan hasil training resmi
pada Sub-bab 4.5.1 skripsi, yaitu order (1,1,0) dengan seasonal order
(1,1,0,12).

**Cara pakai:**
1. Pastikan berkas `dataset_toko.csv` berada di folder yang sama dengan notebook ini,
   atau ubah variabel `csv_path` di Sel 1 sesuai lokasi berkas Anda.
2. Jalankan seluruh sel dari atas ke bawah secara berurutan.
3. Ganti `INDEX_PRODUK` pada Sel 5 apabila ingin memverifikasi produk lain.


## Sel 1 — Setup: Import Library dan Konstanta

In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

csv_path = "dataset_toko.csv"   # <-- ubah jika lokasi berkas berbeda

# Konstanta -- identik dengan prediction.py
SPLIT_PCT        = 0.80
MIN_BULAN        = 15
MIN_BULAN_ARIMA  = 20
N_WINDOW_ARIMA   = 24

ARIMA_ORDERS = [
    (1,1,1), (1,1,0), (0,1,1), (2,1,0),
    (0,1,2), (2,1,1), (1,1,2), (3,1,0),
    (0,1,3), (2,1,2), (1,2,1), (0,2,1),
]
ARIMA_SEASONAL_ORDERS = [(0,0,0,0), (1,1,0,12)]

KALENDER_LIBUR = {
    '2020-05': 0.40, '2021-05': 0.45, '2022-05': 0.55,
    '2023-04': 0.50, '2024-04': 0.65, '2025-03': 0.90,
    '2020-06': 0.60, '2021-06': 0.60, '2022-06': 0.65,
    '2020-07': 0.60, '2021-07': 0.65, '2022-07': 0.70,
    '2023-06': 0.65, '2024-05': 0.60,
    '2024-06': 0.12, '2024-07': 0.28,
    '2024-08': 0.55, '2024-12': 0.40,
    '2025-01': 0.58,
}
BULAN_EKSKLUDE = ['2024-06', '2024-07']

print(f"Total kombinasi grid search per produk: {len(ARIMA_ORDERS)} x {len(ARIMA_SEASONAL_ORDERS)} = "
      f"{len(ARIMA_ORDERS) * len(ARIMA_SEASONAL_ORDERS)}")


Total kombinasi grid search per produk: 12 x 2 = 24


## Sel 2 — Load dan Preprocessing Data (identik dengan Tahap 1-9 pada notebook utama)

In [2]:
df = pd.read_csv(csv_path, on_bad_lines='skip')
df['Tanggal Pembayaran'] = pd.to_datetime(df['Tanggal Pembayaran'], format='mixed', errors='coerce')
df = df.dropna(subset=['Tanggal Pembayaran'])
df = df[df['Status Terakhir'] == 'Pesanan Selesai'].copy()

for col in ['Harga Jual (IDR)', 'Jumlah Produk Dibeli']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

df['item_revenue'] = (df['Harga Jual (IDR)'] * df['Jumlah Produk Dibeli']).clip(lower=0)
df['bulan_period'] = df['Tanggal Pembayaran'].dt.to_period('M')
df['bulan']        = df['Tanggal Pembayaran'].dt.month

bulan_list  = sorted(df['bulan_period'].unique())
split_idx   = int(len(bulan_list) * SPLIT_PCT)
bulan_train = bulan_list[:split_idx]

monthly_all = (
    df.groupby(['bulan_period', 'bulan', 'Nama Produk'])
    .agg(qty=('Jumlah Produk Dibeli', 'sum'), revenue=('item_revenue', 'sum'))
    .reset_index()
    .sort_values(['Nama Produk', 'bulan_period'])
)
monthly_train = monthly_all[monthly_all['bulan_period'].isin(bulan_train)]

produk_count   = monthly_train.groupby('Nama Produk')['bulan_period'].count()
produk_arima_l = produk_count[produk_count >= MIN_BULAN_ARIMA].index.tolist()

print(f"Total bulan training : {len(bulan_train)}")
print(f"Produk layak ARIMA (riwayat >= {MIN_BULAN_ARIMA} bulan): {len(produk_arima_l)}")


Total bulan training : 47
Produk layak ARIMA (riwayat >= 20 bulan): 171


## Sel 3 — Pilih Produk yang Akan Diverifikasi

In [3]:
INDEX_PRODUK = 0   # ganti angka ini untuk memverifikasi produk lain (0 s.d. len(produk_arima_l)-1)

p_verifikasi = produk_arima_l[INDEX_PRODUK]

df_c = (monthly_train[monthly_train['Nama Produk'] == p_verifikasi]
        .sort_values('bulan_period'))
df_c = df_c[~df_c['bulan_period'].astype(str).isin(BULAN_EKSKLUDE)].tail(N_WINDOW_ARIMA)

fl_v   = df_c['bulan_period'].astype(str).map(lambda x: KALENDER_LIBUR.get(x, 1.0)).values
ts_log = np.log1p(df_c['qty'].clip(lower=0.1).values / np.maximum(fl_v, 0.1))

print(f"Produk yang diverifikasi : {p_verifikasi}")
print(f"Jumlah observasi (bulan) : {len(ts_log)}")
df_c[['bulan_period', 'qty', 'revenue']].reset_index(drop=True)


Produk yang diverifikasi : AS 6mm KIPAS ANGIN KECIL 7 IN 9 IN BATANG BESI
Jumlah observasi (bulan) : 24


,bulan_period,qty,revenue
0,2022-04,32.0,213200.0
1,2022-05,16.0,110400.0
2,2022-06,10.0,65000.0
3,2022-07,7.0,48300.0
4,2022-08,13.0,85700.0
5,2022-09,15.0,103500.0
6,2022-10,18.0,124200.0
7,2022-11,29.0,182600.0
8,2022-12,30.0,203000.0
9,2023-01,25.0,168500.0


## Sel 4 — Grid Search AIC Manual (24 Kombinasi) dengan Verifikasi Rumus

Untuk setiap kombinasi `(base_order, seasonal_order)`, sel ini:
1. Melakukan fitting model SARIMAX (identik dengan Kode 4.13).
2. Mengambil `k` (jumlah parameter, `model.df_model`) dan log-likelihood (`model.llf`).
3. Menghitung AIC secara manual: `AIC_manual = 2k - 2*ln(L)`.
4. Membandingkan `AIC_manual` dengan atribut `model.aic` bawaan statsmodels.


In [4]:
hasil_grid = []
best_aic_verif = np.inf
best_order_verif = None
best_seasonal_verif = None

for base_ord in ARIMA_ORDERS:
    for seas_ord in ARIMA_SEASONAL_ORDERS:
        try:
            model = SARIMAX(
                ts_log, order=base_ord, seasonal_order=seas_ord,
                enforce_stationarity=False, enforce_invertibility=False
            ).fit(disp=False)

            aic_value      = model.aic
            k_params       = model.df_model
            log_likelihood = model.llf

            aic_manual = 2 * k_params - 2 * log_likelihood
            cocok = abs(aic_manual - aic_value) < 1e-6

            hasil_grid.append({
                "base_order": base_ord, "seasonal_order": seas_ord,
                "k_parameter": k_params,
                "log_likelihood": round(log_likelihood, 3),
                "AIC_statsmodels": round(aic_value, 3),
                "AIC_manual_2k_2lnL": round(aic_manual, 3),
                "Rumus_Cocok": "YA" if cocok else "TIDAK"
            })

            if aic_value < best_aic_verif:
                best_aic_verif = aic_value
                best_order_verif = base_ord
                best_seasonal_verif = seas_ord
        except Exception:
            hasil_grid.append({
                "base_order": base_ord, "seasonal_order": seas_ord,
                "k_parameter": None, "log_likelihood": None,
                "AIC_statsmodels": None, "AIC_manual_2k_2lnL": None,
                "Rumus_Cocok": "GAGAL FIT"
            })

df_verif = pd.DataFrame(hasil_grid)
df_verif_valid = df_verif[df_verif["Rumus_Cocok"] != "GAGAL FIT"].sort_values("AIC_statsmodels")

print(f"Kombinasi berhasil di-fit : {len(df_verif_valid)} dari {len(df_verif)}")
print(f"Kombinasi rumus TIDAK cocok (harus 0): "
      f"{(df_verif_valid['Rumus_Cocok'] == 'TIDAK').sum()}")
df_verif_valid.reset_index(drop=True)


Kombinasi berhasil di-fit : 24 dari 24
Kombinasi rumus TIDAK cocok (harus 0): 0


,base_order,seasonal_order,k_parameter,log_likelihood,AIC_statsmodels,AIC_manual_2k_2lnL,Rumus_Cocok
0,"(1, 1, 0)","(1, 1, 0, 12)",3,0.000,6.000,6.000,YA
1,"(0, 1, 1)","(1, 1, 0, 12)",3,0.000,6.000,6.000,YA
2,"(0, 2, 1)","(1, 1, 0, 12)",3,0.000,6.000,6.000,YA
3,"(2, 1, 0)","(1, 1, 0, 12)",4,0.000,8.000,8.000,YA
4,"(0, 1, 2)","(1, 1, 0, 12)",4,0.000,8.000,8.000,YA
5,"(1, 1, 1)","(1, 1, 0, 12)",4,0.000,8.000,8.000,YA
6,"(1, 2, 1)","(1, 1, 0, 12)",4,0.000,8.000,8.000,YA
7,"(1, 1, 2)","(1, 1, 0, 12)",5,0.000,10.000,10.000,YA
8,"(2, 1, 1)","(1, 1, 0, 12)",5,0.000,10.000,10.000,YA
9,"(0, 1, 3)","(1, 1, 0, 12)",5,0.000,10.000,10.000,YA


## Sel 5 — Kesimpulan: Kombinasi Terbaik dan Perbandingan

In [5]:
print("=" * 70)
print("KOMBINASI TERBAIK HASIL VERIFIKASI ULANG")
print("=" * 70)
print(f"  base_order      = {best_order_verif}")
print(f"  seasonal_order  = {best_seasonal_verif}")
print(f"  AIC             = {best_aic_verif:.3f}")
print()
print("Interpretasi:")
print("Kombinasi ini menghasilkan keseimbangan terbaik antara kecocokan")
print("model terhadap data (log-likelihood tinggi) dan kompleksitas model")
print("(jumlah parameter tidak berlebihan), sesuai prinsip parsimony yang")
print("mendasari kriteria AIC.")
print()
print("Bandingkan nilai base_order dan seasonal_order di atas dengan hasil")
print("best_orders_arima dan best_seasonal_arima pada notebook Tahap 13")
print("(Training ARIMA per Produk) untuk produk yang sama, untuk memastikan")
print("konsistensi antara hasil verifikasi ini dengan hasil training resmi")
print("yang dilaporkan pada Sub-bab 4.5.1 skripsi.")


KOMBINASI TERBAIK HASIL VERIFIKASI ULANG
  base_order      = (1, 1, 0)
  seasonal_order  = (1, 1, 0, 12)
  AIC             = 6.000

Interpretasi:
Kombinasi ini menghasilkan keseimbangan terbaik antara kecocokan
model terhadap data (log-likelihood tinggi) dan kompleksitas model
(jumlah parameter tidak berlebihan), sesuai prinsip parsimony yang
mendasari kriteria AIC.

Bandingkan nilai base_order dan seasonal_order di atas dengan hasil
best_orders_arima dan best_seasonal_arima pada notebook Tahap 13
(Training ARIMA per Produk) untuk produk yang sama, untuk memastikan
konsistensi antara hasil verifikasi ini dengan hasil training resmi
yang dilaporkan pada Sub-bab 4.5.1 skripsi.
